# Imports and Configuration

In [1]:
!pip install scikit-surprise
!pip install implicit
from implicit.als import AlternatingLeastSquares
import re, gc, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from surprise import SVD, Dataset, Reader

warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 19.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


config

In [3]:
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)

MIN_BOOK_RATINGS = 100
MIN_USER_RATINGS = 20
N_USERS = 100000                # number of sample (1M is too large -> not enough RAM -> explosion)
RELEVANT_RATING = 4             # threshold for "liked"
K = 10                          # top-K for Precision@K / Recall@K
EVAL_USERS = 1000
LIKED_LIMIT = 50
N_NEIGHBORS = 100
USER_NEIGHBORS = 200
RANDOM_STATE = 67

rng = np.random.default_rng(RANDOM_STATE)

### Load Data

In [4]:
interactions = pd.read_parquet("/content/drive/MyDrive/USTH/Fundamental of DS/dataset/50ubr.parquet")

book_cols = ["book_id", "title", "author_names"] + ["genre_1", "genre_2", "genre_3", "genre_4", "genre_5"] + ["average_rating", "ratings_count"]
books = pd.read_parquet("/content/drive/MyDrive/USTH/Fundamental of DS/dataset/mart_book_statistics.parquet", columns=book_cols).drop_duplicates("book_id")

print(f"Interactions: {interactions.shape}")
print(f"Books: {books.shape}")

Interactions: (50000000, 3)
Books: (2363585, 10)


### Data Cleaning

In [5]:
df = interactions[interactions["book_id"].isin(books["book_id"])]
df = df.drop_duplicates(["user_id", "book_id"])
del interactions
gc.collect()

0

In [6]:
book_counts = df["book_id"].value_counts()
df = df[df["book_id"].isin(book_counts[book_counts >= MIN_BOOK_RATINGS].index)]

user_counts = df["user_id"].value_counts()
users_clean = user_counts.index.to_numpy()
if len(users_clean) > N_USERS:
    users_clean = rng.choice(users_clean, N_USERS, replace=False)
df = df[df["user_id"].isin(users_clean)].copy()

In [7]:
# convert author to list of strings
def to_list(x):
    if x is None:
        return []
    if isinstance(x, str):
        return [x]
    try:
        return [str(a) for a in x if a is not None]
    except TypeError:
        return []

# matrix
df["user_idx"], user_ids = pd.factorize(df["user_id"], sort=True)
df["item_idx"], book_ids = pd.factorize(df["book_id"], sort=True)
df["rating"] = df["rating"].astype("float32")
n_users, n_items = len(user_ids), len(book_ids)

books_meta = books.set_index("book_id").loc[book_ids].copy()
books_meta["author"] = books_meta["author_names"].apply(lambda x: ", ".join(to_list(x)))
del books
gc.collect()

print(f"Users: {n_users:,}")
print(f"Books: {n_items:,}")
print(f"Ratings: {len(df):,}\n")
print(f"Matrix Density: {len(df) / (n_users * n_items):.4%}\n")
print(df["rating"].value_counts().sort_index().to_string())

Users: 100,000
Books: 46,151
Ratings: 8,331,095

Matrix Density: 0.1805%

rating
1.0     169093
2.0     471801
3.0    1750863
4.0    2962862
5.0    2976476


# Model

### Train / validation / test split

Split per user

Example: (User has 20 rating -> 16 train / 2 val / 2 test)

In [8]:
df = df.sample(frac=1, random_state=RANDOM_STATE)
position = df.groupby("user_idx").cumcount()
size = df.groupby("user_idx")["user_idx"].transform("size")
frac     = position / size

train = df[frac < 0.8]
val = df[(frac >= 0.8) & (frac < 0.9)]
test = df[frac >= 0.9]

print(f"Train: {len(train):,}")
print(f"Valid: {len(val):,}")
print(f"Test: {len(test):,}")

Train: 6,704,859
Valid: 838,099
Test: 788,137


In [9]:
# user-item matrix from train
R_train   = csr_matrix((train["rating"], (train["user_idx"], train["item_idx"])), shape=(n_users, n_items))
item_user = R_train.T.tocsr()

### Evaluation setup

In [10]:
def group_items(data):
    return data.groupby("user_idx")["item_idx"].agg(list).to_dict()

# group book that user like (rating > 4)
liked_train = group_items(train[train["rating"] >= RELEVANT_RATING])
liked_val = group_items(val[val["rating"] >= RELEVANT_RATING])
liked_test = group_items(test[test["rating"] >= RELEVANT_RATING])

# eligible user = user that in both liked train/val/test
eligible   = [u for u in liked_train if u in liked_val and u in liked_test]
eval_users = rng.choice(eligible, min(EVAL_USERS, len(eligible)), replace=False)

# books that user has readed
sub = lambda d: d[d["user_idx"].isin(eval_users)]
readed_train = group_items(sub(train))
readed_val = group_items(sub(val))
readed_for_val = {u: readed_train[u] for u in eval_users}
readed_for_test = {u: readed_train[u] + readed_val[u] for u in eval_users}

In [11]:
def user_metrics(scores, u):
    out = []
    for seen, relevant in ((readed_for_val[u], liked_val[u]), (readed_for_test[u], liked_test[u])):
        s = scores.astype("float64").copy()
        s[seen] = -np.inf                              # already-seen books -> minus score
        top = np.argpartition(-s, K)[:K]
        hits = np.isin(top, relevant).sum()
        out += [hits / K, hits / len(relevant)]
    return np.array(out)

def evaluate(score_fn):
    return np.mean([user_metrics(score_fn(u), u) for u in eval_users], axis=0)

def rating_errors(pred, true):
    err = np.asarray(pred) - np.asarray(true)
    return np.abs(err).mean(), np.sqrt((err ** 2).mean())

results = {}
def save(name, m, mae=None, rmse=None):
    results[name] = dict(val_P=m[0], val_R=m[1], test_P=m[2], test_R=m[3], MAE=mae, RMSE=rmse)

## **Popularity Baseline**

In [12]:
stats = train.groupby("item_idx")["rating"].agg(["count", "mean"]).reindex(range(n_items))
v = stats["count"].fillna(0).to_numpy()             # number of rating of the book
R = stats["mean"].fillna(0).to_numpy()              # avg rating of the book
C = train["rating"].mean()                          # avg rating of all the books
m = np.percentile(v[v > 0], 80)                     # minimum votes threshold

pop_score = v / (v + m) * R + m / (v + m) * C

mae, rmse = rating_errors(pop_score[test["item_idx"]], test["rating"])

def pop_fn(user):
    return pop_score

save("Popularity", evaluate(pop_fn), mae, rmse)
pd.Series(results["Popularity"])

,0
val_P,0.004300
val_R,0.006587
test_P,0.003900
test_R,0.006487
MAE,0.752101
RMSE,0.951004


## **Item-Based Collaborative Filtering**

Idea:  user: like books -> recommendations most similar books (cosine similarity)

In [13]:
def make_neighbor_fn(X, nn):

    def score_fn(u):
        seeds = liked_train.get(u, [])[:LIKED_LIMIT]        # take limit 20 books that user like
        scores = np.zeros(X.shape[0])
        if len(seeds) == 0:
            return scores

        dist, idx = nn.kneighbors(X[seeds], n_neighbors=N_NEIGHBORS)
        scores += np.bincount(idx.ravel(), weights=(1 - dist).ravel(), minlength=X.shape[0])
        return scores

    return score_fn

L_train = (R_train >= RELEVANT_RATING).astype("float32").tocsr()
item_user_like = L_train.T.tocsr()

nn_cf = NearestNeighbors(metric="cosine", algorithm="brute").fit(item_user_like)
itemcf_fn = make_neighbor_fn(item_user_like, nn_cf)

save("Item-CF", evaluate(itemcf_fn))
pd.Series(results["Item-CF"])

,0
val_P,0.085100
val_R,0.174074
test_P,0.081100
test_R,0.164747
MAE,NaN
RMSE,NaN


## **User-Based Collaborative Filtering**

In [14]:
USER_NEIGHBORS = 200

nn_user = NearestNeighbors(metric="cosine", algorithm="brute").fit(L_train)

def usercf_fn(u):
    dist, idx = nn_user.kneighbors(L_train[u], n_neighbors=USER_NEIGHBORS + 1)
    dist, idx = dist[0], idx[0]
    keep = idx != u
    sims = (1 - dist[keep])[:USER_NEIGHBORS] ** 2
    neighbours = idx[keep][:USER_NEIGHBORS]
    return L_train[neighbours].T @ sims

save("User-CF", evaluate(usercf_fn))
pd.Series(results["User-CF"])

,0
val_P,0.092500
val_R,0.180664
test_P,0.093900
test_R,0.187979
MAE,NaN
RMSE,NaN


## **SVD Matrix Factorization**

In [15]:
reader   = Reader(rating_scale=(1, 5))
trainset = Dataset.load_from_df(train[["user_idx", "item_idx", "rating"]], reader).build_full_trainset()
val_list  = list(zip(val["user_idx"].tolist(),  val["item_idx"].tolist(),  val["rating"].tolist()))
test_list = list(zip(test["user_idx"].tolist(), test["item_idx"].tolist(), test["rating"].tolist()))

def svd_errors(algo, data):
    est = np.array([p.est for p in algo.test(data)])
    return rating_errors(est, [r for _, _, r in data])

svd_models, rows = {}, []
for f in [20, 50, 100]:
    algo = SVD(n_factors=f, n_epochs=15, random_state=RANDOM_STATE)
    algo.fit(trainset)
    mae_v, rmse_v = svd_errors(algo, val_list)
    svd_models[f] = algo
    rows.append({"n_factors": f, "val_MAE": mae_v, "val_RMSE": rmse_v})

svd_table = pd.DataFrame(rows)
best_f = int(svd_table.loc[svd_table["val_RMSE"].idxmin(), "n_factors"])
print(svd_table.round(4).to_string(index=False))
print("selected n_factors (lowest validation RMSE):", best_f)

 n_factors  val_MAE  val_RMSE
        20   0.6585    0.8458
        50   0.6568    0.8445
       100   0.6559    0.8441
selected n_factors (lowest validation RMSE): 100


In [16]:
svd = svd_models[best_f]
mae, rmse = svd_errors(svd, test_list)

raw_items = [int(trainset.to_raw_iid(i)) for i in range(trainset.n_items)]
def svd_fn(u):
    s = svd.bi + svd.qi @ svd.pu[trainset.to_inner_uid(int(u))]
    full = np.full(n_items, s.min())
    full[raw_items] = s
    return full

save("SVD", evaluate(svd_fn), mae, rmse)
pd.Series(results["SVD"])

,0
val_P,0.000600
val_R,0.001114
test_P,0.000100
test_R,0.000167
MAE,0.656455
RMSE,0.844879


In [17]:
W = R_train.copy().tocsr()
W.data = np.where(W.data >= RELEVANT_RATING, 3.0, 1.0).astype("float32")   # confidence

als = AlternatingLeastSquares(factors=64, regularization=0.05,
                              iterations=15, random_state=RANDOM_STATE)
als.fit(W)                                   # W shape: (n_users, n_items)

U, V = als.user_factors, als.item_factors    # latent vectors of users / books
if hasattr(U, "to_numpy"):                   # in case the GPU version is used
    U, V = U.to_numpy(), V.to_numpy()

def als_fn(u):
    return V @ U[u]                          # score = dot product of user and every book

save("ALS", evaluate(als_fn))
pd.Series(results["ALS"])

  0%|          | 0/15 [00:00<?, ?it/s]

,0
val_P,0.100100
val_R,0.188929
test_P,0.104500
test_R,0.209461
MAE,NaN
RMSE,NaN


## **Content-Based Filtering**

In [18]:
def clean(token):
    return re.sub(r"\W+", "_", str(token).lower()).strip("_")

genre_text = books_meta[["genre_1", "genre_2", "genre_3", "genre_4", "genre_5"]].apply(
    lambda row: " ".join(clean(g) for g in row if pd.notna(g) and clean(g)), axis=1)

author_text = books_meta["author_names"].apply(
    lambda x: " ".join(t for t in (clean(a) for a in to_list(x)) if t))

def build_content_fn(texts):
    X  = TfidfVectorizer(token_pattern=r"\S+", lowercase=False).fit_transform(texts)
    nn = NearestNeighbors(metric="cosine", algorithm="brute").fit(X)
    return make_neighbor_fn(X, nn), X.shape[1]

### Genre

In [19]:
genre_fn, size = build_content_fn(genre_text)
save("CB (Genre)", evaluate(genre_fn))
print("vocabulary size:", size); pd.Series(results["CB (Genre)"])

vocabulary size: 16


,0
val_P,0.001000
val_R,0.001220
test_P,0.000500
test_R,0.000747
MAE,NaN
RMSE,NaN


### Author

In [20]:
author_fn, size = build_content_fn(author_text)
save("CB (Author)", evaluate(author_fn))
print("vocabulary size:", size); pd.Series(results["CB (Author)"])

vocabulary size: 18784


,0
val_P,0.014900
val_R,0.032076
test_P,0.013100
test_R,0.030937
MAE,NaN
RMSE,NaN


### Genre + Author

In [21]:
both_fn, size = build_content_fn(genre_text + " " + author_text)
save("CB (Genre + Author)", evaluate(both_fn))
print("vocabulary size:", size); pd.Series(results["CB (Genre + Author)"])

vocabulary size: 18800


,0
val_P,0.012200
val_R,0.031066
test_P,0.009800
test_R,0.024911
MAE,NaN
RMSE,NaN


# Model Comparison

In [22]:
def comparison_table(split="test"):
    rows = []
    for name, r in results.items():
        f = lambda x: "-" if x is None else round(float(x), 4)
        rows.append({"Model": name, "MAE": f(r["MAE"]), "RMSE": f(r["RMSE"]),
                     "Precision@10": f(r[f"{split}_P"]), "Recall@10": f(r[f"{split}_R"])})
    return pd.DataFrame(rows).set_index("Model")

In [23]:
comparison_table("val")

,MAE,RMSE,Precision@10,Recall@10
Model,,,,
Popularity,0.7521,0.951,0.0043,0.0066
Item-CF,-,-,0.0851,0.1741
User-CF,-,-,0.0925,0.1807
SVD,0.6565,0.8449,0.0006,0.0011
ALS,-,-,0.1001,0.1889
CB (Genre),-,-,0.0010,0.0012
CB (Author),-,-,0.0149,0.0321
CB (Genre + Author),-,-,0.0122,0.0311


In [24]:
comparison_table("test")

,MAE,RMSE,Precision@10,Recall@10
Model,,,,
Popularity,0.7521,0.951,0.0039,0.0065
Item-CF,-,-,0.0811,0.1647
User-CF,-,-,0.0939,0.1880
SVD,0.6565,0.8449,0.0001,0.0002
ALS,-,-,0.1045,0.2095
CB (Genre),-,-,0.0005,0.0007
CB (Author),-,-,0.0131,0.0309
CB (Genre + Author),-,-,0.0098,0.0249


## **Hybrid Model**

In [25]:
# Scale score to 0-1 per user
def normalize(s):
    return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else np.zeros_like(s, dtype="float64")

# (user_cf, item_cf, content, als) - each tuple sums to 1
weights = [
    (0.30, 0.20, 0.10, 0.40),
    (0.25, 0.15, 0.10, 0.50),
    (0.40, 0.20, 0.10, 0.30),
    (0.30, 0.10, 0.10, 0.50),
    (0.20, 0.20, 0.10, 0.50),
    (0.35, 0.25, 0.10, 0.30),
]

sums = {w: np.zeros(4) for w in weights}

for u in eval_users:
    user_cf = normalize(usercf_fn(u))
    item_cf = normalize(itemcf_fn(u))
    content = normalize(both_fn(u))
    als_s   = normalize(als_fn(u))
    for w in weights:
        user_w, item_w, content_w, als_w = w
        hybrid_score = user_w * user_cf + item_w * item_cf + content_w * content + als_w * als_s
        sums[w] += user_metrics(hybrid_score, u)

names = [f"U{w[0]}_I{w[1]}_C{w[2]}_A{w[3]}" for w in weights]
hybrid_table = pd.DataFrame(
    [sums[w] / len(eval_users) for w in weights],
    index=names,
    columns=["val_P@10", "val_R@10", "test_P@10", "test_R@10"]
)
print(hybrid_table.round(4).to_string())

# choose weights by VALIDATION (not test)
best_key = hybrid_table["val_P@10"].idxmax()
best_weight = weights[names.index(best_key)]
print("\nSelected weights:", best_key)

save(f"Hybrid ({best_key})", hybrid_table.loc[best_key].to_numpy())

comparison_table("test")

                       val_P@10  val_R@10  test_P@10  test_R@10
U0.3_I0.2_C0.1_A0.4      0.1100    0.2135     0.1131     0.2262
U0.25_I0.15_C0.1_A0.5    0.1105    0.2131     0.1149     0.2274
U0.4_I0.2_C0.1_A0.3      0.1076    0.2100     0.1107     0.2193
U0.3_I0.1_C0.1_A0.5      0.1098    0.2132     0.1150     0.2283
U0.2_I0.2_C0.1_A0.5      0.1109    0.2142     0.1137     0.2260
U0.35_I0.25_C0.1_A0.3    0.1066    0.2080     0.1105     0.2201

Selected weights: U0.2_I0.2_C0.1_A0.5


,MAE,RMSE,Precision@10,Recall@10
Model,,,,
Popularity,0.7521,0.951,0.0039,0.0065
Item-CF,-,-,0.0811,0.1647
User-CF,-,-,0.0939,0.1880
SVD,0.6565,0.8449,0.0001,0.0002
ALS,-,-,0.1045,0.2095
CB (Genre),-,-,0.0005,0.0007
CB (Author),-,-,0.0131,0.0309
CB (Genre + Author),-,-,0.0098,0.0249
Hybrid (U0.2_I0.2_C0.1_A0.5),-,-,0.1137,0.2260


In [26]:
print("Best weight:", best_weight)

Best weight: (0.2, 0.2, 0.1, 0.5)


## 12. Final Recommendation Demo

In [27]:
user_to_idx = {uid: i for i, uid in enumerate(user_ids)}

all_rated = df.groupby("user_idx")["item_idx"].agg(list).to_dict()

user_w, item_w, content_w, als_w = best_weight


def hybrid_scores(u):
    user_cf = normalize(usercf_fn(u))
    item_cf = normalize(itemcf_fn(u))
    content = normalize(both_fn(u))
    als_s = normalize(als_fn(u))

    return (
        user_w * user_cf
        + item_w * item_cf
        + content_w * content
        + als_w * als_s
    )


def recommend_books(user_id, n=10):

    # Unknown user -> popularity fallback
    if user_id not in user_to_idx:
        scores = pop_score.copy()
        seen = []

    else:
        u = user_to_idx[user_id]
        scores = hybrid_scores(u)
        seen = all_rated.get(u, [])

    scores = scores.astype("float64")

    # Remove books already rated
    scores[seen] = -np.inf

    # Top-N
    top = np.argsort(-scores)[:n]

    info = books_meta.iloc[top]

    recommend10 = pd.DataFrame({
        "Rank": range(1, n + 1),
        "book_id": book_ids[top],
        "Title": info["title"].values,
        "Author": info["author"].values,
        "Score": scores[top].round(3),
        "Avg Rating": info["average_rating"].values,
        "Ratings Count": info["ratings_count"].values
    })

    return recommend10

In [28]:
user = user_ids[eval_users[1]]

print(f"\nTop-10 recommendations for user {user}:")
recommend_books(user, n=10)


Top-10 recommendations for user 50fb3f2b301a48a4507a5c6958757ec8:


,Rank,book_id,Title,Author,Score,Avg Rating,Ratings Count
0,1,6148028,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.656,4.36,4333414.0
1,2,48855,The Diary of a Young Girl,"Anne Frank, B.M. Mooyaart-Doubleday, Eleanor R...",0.493,4.20,4331345.0
2,3,2657,To Kill a Mockingbird,Harper Lee,0.449,4.26,7123019.0
3,4,960,"Angels & Demons (Robert Langdon, #1)",Dan Brown,0.400,3.96,3460602.0
4,5,11735983,"Insurgent (Divergent, #2)",Veronica Roth,0.396,4.07,849014.0
5,6,256683,"City of Bones (The Mortal Instruments, #1)",Cassandra Clare,0.382,4.12,1181693.0
6,7,30183,"Marked (House of Night, #1)","Kristin Cast, P.C. Cast",0.370,3.79,364758.0
7,8,1582996,"City of Ashes (The Mortal Instruments, #2)",Cassandra Clare,0.367,4.21,550237.0
8,9,11870085,The Fault in Our Stars,John Green,0.356,4.12,5883396.0
9,10,2866718,"Untamed (House of Night, #4)","Kristin Cast, P.C. Cast",0.354,3.95,192869.0


# other